In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')

# Electrification Inequality and Spatial Infrastructure: A Tile-Level Panel Analysis of Brazil, China, and Morocco (2014–2023)

**Authors:** Bouchra Daddaoui · Namuunzaya Barsbold · Amanda González Mejía  
**Repository:** https://github.com/namibarsbold/STATS201-Course-project  
**GitHub Pages:** https://namibarsbold.github.io/STATS201-Course-project/

---
## 1. Research Question and Motivation

This project asks how population density and infrastructure jointly predict satellite-observed nighttime light intensity, and whether the population–light relationship varies across regions and countries in ways consistent with spatial electrification inequality.

Nighttime lights are widely used as a remotely sensed proxy for economic activity and development when administrative statistics are noisy or missing; related work emphasizes that lights also reflect infrastructure access and spatial inequality, capturing uneven electrification, industrial concentration, and development disparities (Henderson et al., 2012; Falchetta et al., 2023).

Electricity is a foundational input shaping labor markets, productivity, and household welfare. Causal evidence from rural electrification expansions links electricity access to higher labor market participation, particularly increased female employment in Africa (Dinkelman, 2011). If lights proxy electrification, then spatial variation in lights plausibly reflects differences in economic opportunity.

This variation is spatially structured. Subnational light patterns have been used to measure within-country inequality, revealing regional disparities that national averages obscure (Lipscomb, 2013). Related work argues that population–development elasticities vary across spatial contexts as institutions and infrastructure change how demographic concentration translates into observed outcomes (Lessmann & Seidel, 2017).

**Success** looks like a model that captures substantial variance in nighttime light intensity, identifies regime-specific population–light elasticities, and provides interpretable evidence on how spatial structure mediates the relationship between population density and electrification access.

---
## 2. Data

### 2.1 Sources

We construct a tile-level panel for three country case studies — **Brazil, China, and Morocco** — covering 2014–2023.

**Nighttime lights** come from annual composites of the VIIRS Day/Night Band produced for low-light detection by NASA and NOAA, distributed via the Earth Observation Group at Colorado School of Mines. These products are processed composites (not raw photographs): annual composites filter sunlit/moonlit/cloudy observations, screen ephemeral lights and background noise, rely on the "vcm" configuration to exclude stray-light-impacted data, and apply outlier filtering to reduce biomass burning and other anomalous radiance.

**Population data** are annual gridded estimates from WorldPop, produced via spatial disaggregation (often Random Forest–based dasymetric mapping) rather than direct census counts at each pixel — a key measurement limitation.

**Infrastructure features** are derived from tile geometry (distance to urban core, local urban density, centrality). Road density from OpenStreetMap via Geofabrik was initially planned but excluded from the final specification due to processing and harmonization constraints; it is treated as future work.

### 2.2 Unit of Analysis

The unit of observation is a **tile × year** observation on a consistent spatial grid within each country. Rasters are aggregated into fixed spatial tiles using adaptive tile sizing (256–1024 pixels; 60–80 tiles along the short dimension) to manage computation while preserving within-country panel structure. Tiles are filtered for data quality by excluding observations below a valid-pixel threshold (< 500 valid pixels).

The resulting dataset contains **30,442 tile-year observations** across the three countries. While the dataset spans 2014–2023, the final regression results focus on the **2023 cross-section** to enable clean cross-country comparison.

### 2.3 Key Variables

| Variable | Role | Description |
|----------|------|-------------|
| $\log(1 + \text{Light}_{it})$ | Target | Log-transformed mean annual radiance (nW/cm²/sr) per tile |
| $\log(1 + \text{Pop}_{it})$ | Core feature | Log-transformed WorldPop population estimate |
| Region type $R_{it}$ | Feature | Bivariate typology: *urban_core*, *dense_dim*, *bright_sparse*, *mixed*, *empty_or_rural* |
| $\log(\text{DistanceToUrbanCore}_{it})$ | Infrastructure proxy | Log distance to the nearest urban-core tile |
| $\log(\text{LocalUrbanDensity}_{it})$ | Infrastructure proxy | Log count of urban-classified tiles within a fixed radius |
| $\text{Centrality}_{it}$ | Infrastructure proxy | Distance from tile centroid to national geographic centroid |

**Region typology** is constructed using country-year–specific median thresholds on population density and light intensity:
- **urban_core**: high population, high light  
- **dense_dim**: high population, low light  
- **bright_sparse**: low population, high light  
- **mixed**: intermediate combinations near thresholds  
- **empty_or_rural**: low population, low light

Thresholds are computed separately for each country-year. The typology is fully data-driven and should be interpreted as descriptive stratification rather than a causal treatment.

### 2.4 Preprocessing

Nighttime light rasters and WorldPop population rasters are aggregated into fixed spatial tiles to form a panel. Key preprocessing steps include:

1. **Tile construction** — adaptive tile sizing (256–1024 pixels; 60–80 tiles along the short axis per country)
2. **Quality filtering** — tiles with fewer than 500 valid pixels excluded
3. **Log transformations** — $\log(1+x)$ applied to both radiance and population to handle zeros and reduce skew
4. **Region classification** — bivariate median-threshold scheme applied per country-year
5. **Infrastructure variables** — computed from tile geometry (distances, neighbor counts)

In [ ]:
# EDA: Distribution of nighttime light radiance
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(plt.imread('../figures/fig_hist_mean_rad.png'))
axes[0].axis('off')
axes[0].set_title('Figure 1a — Distribution of Mean Radiance', fontsize=10)
axes[1].imshow(plt.imread('../figures/fig_scatter_mean_vs_sd.png'))
axes[1].axis('off')
axes[1].set_title('Figure 1b — Mean vs SD of Radiance', fontsize=10)
plt.tight_layout()
plt.show()

### 2.5 Ethical Considerations

Inputs are aggregated satellite imagery and gridded population surfaces that do not identify individuals. However, outputs can still be sensitive if interpreted as performance or deprivation measures; results should be communicated cautiously **without normative labeling of regions**. Cross-country comparisons reflect differences in spatial structure and development trajectories, and should not be read as simple rankings.

WorldPop estimates introduce measurement uncertainty as they are modeled (not census-based) population surfaces. If OpenStreetMap-derived roads are incorporated in future work, attribution and ODbL license compliance should be explicitly acknowledged.

---
## 3. Problem Setup

This project is a **supervised regression** task with the objective to model spatial variation in satellite-observed nighttime light intensity as a function of population density and infrastructure characteristics.

- **ML task:** Cross-sectional regression at the tile level (2023 cross-section)  
- **Target variable:** $\log(1 + \text{Light}_{it})$ — log mean annual radiance per tile  
- **Core features:** $\log(1 + \text{Pop}_{it})$, region-type indicators, population × region interactions, $\log(\text{DistanceToUrbanCore})$, $\log(\text{LocalUrbanDensity})$, Centrality  

**Key assumptions:**
1. Nighttime radiance is a valid proxy for electrification- and development-related activity.
2. The log–log specification yields interpretable elasticities (a 1% increase in population is associated with a β% change in lights).
3. Observations are spatially correlated within countries; inference therefore relies on robust standard errors and descriptive interpretation rather than causal identification.

---
## 4. Methods

### 4.1 Baseline Models

We use **interpretable regression** (rather than high-capacity machine learning) because the goal is estimation and interpretation of regime-specific elasticities linking population density to nighttime light intensity. All models use a log–log structure.

**Pooled Log–Log Model** — establishes the aggregate population–light relationship without allowing spatial heterogeneity:

$$\log(1 + \text{Light}_{it}) = \alpha + \beta \log(1 + \text{Pop}_{it}) + \varepsilon_{it}$$

**Regime-Interaction Model** — allows elasticities to vary across spatial development regimes:

$$\log(1 + \text{Light}_{it}) = \alpha + \beta \log(1 + \text{Pop}_{it}) + \gamma_r + \delta_r \bigl(\log(1 + \text{Pop}_{it}) \times R_{it}\bigr) + \varepsilon_{it}$$

where $r$ indexes region type. The regime-specific elasticity for regime $r$ is $\beta + \delta_r$. This specification captures heterogeneity in how population translates into light across urban cores, dense-but-dim regions, bright-sparse areas, mixed tiles, and empty/rural tiles.

In [ ]:
# Baseline regression table
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(plt.imread('../figures/week7_outputs/week7_baseline_regression_table.png'))
ax.axis('off')
ax.set_title('Table 1 — Baseline OLS Regression Results (2023 Cross-Section)', fontsize=11)
plt.tight_layout()
plt.show()

### 4.2 Final Model

The final model extends the regime-interaction baseline by incorporating spatial accessibility variables that proxy infrastructure conditions. These variables are added additively while retaining regime-specific elasticities:

$$\log(1 + \text{Light}_{it}) = \alpha + (\beta + \delta_r)\log(1 + \text{Pop}_{it}) + \lambda_1 \log(\text{DistanceToUrbanCore}_{it}) + \lambda_2 \log(\text{LocalUrbanDensity}_{it}) + \lambda_3 \text{Centrality}_{it} + \gamma_r + \varepsilon_{it}$$

Models are estimated by OLS. Given heteroskedasticity in residuals (especially in high-light urban regions), we report **heteroskedasticity-robust standard errors**. Key design choices that can affect estimates include tile sizing, valid-pixel thresholds, log(1+x) transforms, and median-based regime thresholds.

In [ ]:
# Final model: infrastructure coefficient estimates
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(plt.imread('../figures/final report/fig2_infra_coefs.png'))
ax.axis('off')
ax.set_title('Figure 2 — Infrastructure Coefficients by Country (Final Model)', fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. Evaluation

### 5.1 Train/Test Split

We evaluate models **in-sample** because the objective is structural interpretation of elasticities rather than predictive generalization. Neighboring tiles are spatially correlated, so a random train/test split would violate independence assumptions and overstate out-of-sample performance.

Spatially blocked cross-validation is the appropriate method for formal out-of-sample evaluation, but it is not implemented in the current analysis and is noted as a limitation. All reported metrics therefore reflect in-sample fit.

### 5.2 Metrics

We report three complementary performance measures:

| Metric | Definition | Why it matches the goal |
|--------|------------|-------------------------|
| **R²** | Share of variance in $\log(1+\text{Light})$ explained by the model | Captures overall explanatory power |
| **RMSE** | Root mean squared error in log units | Measures typical prediction error magnitude |
| **AIC** | Akaike Information Criterion | Compares model efficiency while penalizing additional covariates |

### 5.3 Results

**Baseline Interaction Model** — fit is high across countries (R² = 0.879–0.941), and elasticities vary sharply by regime, indicating systematic spatial nonlinearity in population–light coupling:

| Regime | Brazil | China | Morocco |
|--------|--------|-------|---------|
| empty_or_rural | 0.014 | 0.017 | 0.015 |
| mixed | 0.056 | 0.039 | 0.083 |
| bright_sparse | 0.039 | 0.030 | 0.238*** |
| **urban_core** | **0.471** | **0.696** | **0.544** |

Empty/rural elasticities are near zero: a 10% population increase corresponds to roughly a 0.1–0.2% change in lights. Urban-core elasticities are large — in China, a 10% population increase in urban cores is associated with nearly a 7% increase in lights, consistent with an "urban premium" where dense built form and shared networks amplify the population–light relationship.

**Infrastructure-Augmented Model** — adding distance to urban core, local urban density, and centrality modestly but consistently improves performance:

| Country | ΔR² | ΔAIC |
|---------|-----|------|
| Morocco | +0.013 | −6.7 |
| China | +0.005 | −28.1 |
| Brazil | +0.001 | — |

Distance to urban centers is negative and significant in Brazil (−0.024***) and China (−0.027***). Local urban density is strongly negative in all countries (Morocco −0.749**; Brazil −0.786***; China −1.705***). In Morocco, $\log(\text{pop})$ remains large and highly significant (0.242***) after conditioning on infrastructure, while in Brazil and China the direct population coefficient becomes small and insignificant — consistent with the idea that what appeared as a "population effect" in simpler models is largely mediated by spatial accessibility.

In [ ]:
# R² comparison across models and countries
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(plt.imread('../figures/final report/fig1_r2_comparison.png'))
ax.axis('off')
ax.set_title('Figure 3 — R² Comparison: Baseline vs Final Model by Country', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: predicted vs actual (2023) for each country
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
countries = ['Brazil', 'China', 'Morocco']
labels = ['a', 'b', 'c']
for ax, country, lbl in zip(axes, countries, labels):
    ax.imshow(plt.imread(f'../figures/week7_outputs/scatter_{country}_2023.png'))
    ax.axis('off')
    ax.set_title(f'Figure 4{lbl} — {country} (2023)', fontsize=10)
plt.suptitle('Predicted vs Actual Nighttime Light — 2023 Cross-Section', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Error Analysis

Models capture large-scale spatial structure, but **residual maps show remaining spatial clustering**, suggesting omitted spatial processes.

Residual-versus-fitted plots indicate modest dispersion reduction (especially in Morocco's infrastructure model), though **heteroskedasticity persists in high-light urban regions** (notably China), possibly due to nonlinear dynamics or unobserved metropolitan drivers such as industrial composition, grid architecture, or special economic zones. Robust standard errors address heteroskedasticity in inference, but formal spatial autocorrelation tests (e.g., Moran's I) are needed to fully characterize residual dependence.

Key failure modes:
- **High-light urban cores in China** are systematically under-predicted, suggesting nonlinearity at extreme brightness levels
- **bright_sparse tiles** show pronounced cross-country heterogeneity (Morocco elasticity 0.238 vs Brazil 0.039), implying these tiles capture structurally different phenomena across countries
- **Geographic isolation effects** in Morocco are not fully captured by the population term alone

In [ ]:
# Residual diagnostics
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].imshow(plt.imread('../figures/final report/fig5_residuals.png'))
axes[0].axis('off')
axes[0].set_title('Figure 5a — Residual Distribution', fontsize=10)
axes[1].imshow(plt.imread('../figures/final report/figure_residual_maps_Morocco_2020.png'))
axes[1].axis('off')
axes[1].set_title('Figure 5b — Residual Map: Morocco (2020)', fontsize=10)
plt.tight_layout()
plt.show()

---
## 7. Robustness Check

We identify three key robustness sensitivities:

1. **Region typology** — the data-driven regime classification is constructed from the same variables used in the regression, creating potential endogeneity. Future check: alternative thresholds or base-year regimes.
2. **Aggregation scale** — different countries use different tile sizes, which may affect measured elasticities. Future check: alternative spatial resolutions and harmonized tile sizes.
3. **Non-residential light sources** — industrial sites, gas flares, and road lighting inflate radiance in some tiles independently of electrification. Future check: excluding bright_sparse tiles or trimming extreme radiance values.

The temporal dimension (2014–2023 panel) is not fully exploited in the current cross-sectional specification. Extending to a panel model would allow evaluation of whether population–light elasticities are stable over time or evolving alongside infrastructure expansion. Out-of-sample evaluation requires spatially blocked cross-validation.

In [ ]:
# Robustness: elasticity comparison and temporal R² stability
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].imshow(plt.imread('../figures/final report/fig3_elasticity_comparison.png'))
axes[0].axis('off')
axes[0].set_title('Figure 6a — Elasticity Comparison Across Countries', fontsize=10)
axes[1].imshow(plt.imread('../figures/final report/fig4_temporal_r2.png'))
axes[1].axis('off')
axes[1].set_title('Figure 6b — Temporal R² Stability (2014–2023)', fontsize=10)
plt.tight_layout()
plt.show()

---
## 8. Interpretation and Substantive Takeaways

Population density is strongly associated with nighttime brightness across all three countries. However, the **strength of this association varies systematically by spatial regime**. In dense metropolitan systems, increases in population translate efficiently into greater luminosity, whereas in low-access regions, demographic growth produces little change in brightness.

Cross-country comparisons therefore reflect differences in spatial structure rather than simply differences in national average brightness. For example, China's lower global coupling coefficient masks a sharp internal divide between highly efficient urban centers and weakly responsive rural regions — highlighting how national aggregates obscure regime-level inequality.

Adding infrastructure accessibility variables **improves explanatory power most in geographically polarized systems**, particularly Morocco and China. In Morocco, coastal concentration of lights and large interior distances to urban cores indicate that geographic isolation explains substantial additional variance beyond population. In Brazil and China, once infrastructure is controlled for, the direct population effect largely disappears — suggesting that population density works *through* spatial connectivity rather than independently.

**What readers should not conclude:** These are structural associations, not causal effects. The regime typology is data-driven and endogenous. Nighttime radiance is an imperfect proxy that includes non-residential sources. Results should inform descriptive understanding of spatial inequality rather than direct policy prescription.

**Core finding:** electrification intensity is not purely demographic — it is mediated by spatial access and connectivity. Policymakers targeting electrification gaps should focus not just on population size but on spatial isolation and infrastructure network reach.

In [ ]:
# Infrastructure vs nighttime light — interpretation figure
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].imshow(plt.imread('../figures/final report/fig6_infra_vs_light.png'))
axes[0].axis('off')
axes[0].set_title('Figure 7a — Infrastructure Density vs Nighttime Light', fontsize=10)
axes[1].imshow(plt.imread('../figures/final report/figure_infrastructure_Morocco_2023.png'))
axes[1].axis('off')
axes[1].set_title('Figure 7b — Infrastructure Map: Morocco (2023)', fontsize=10)
plt.tight_layout()
plt.show()

---
## 9. Limitations and Future Work

**Data limitations:**
- WorldPop estimates are modeled population surfaces (Random Forest–based dasymetric mapping), not direct census counts — introducing systematic measurement error.
- Nighttime radiance is an imperfect electrification proxy; it includes industrial sites, gas flares, and road lighting that inflate brightness independently of household electrification.
- Brazil and China GeoTIFF rasters are not included in the repository due to GitHub's 100 MB file size limit (files are 241–394 MB each); they must be re-exported via Google Earth Engine.

**Modeling limitations:**
- Results are structural associations, not causal effects — the regime typology is constructed from the same variables used in regression, creating potential endogeneity.
- The current analysis focuses on the 2023 cross-section; the full 2014–2023 panel is not exploited for dynamic or temporal analysis.
- No spatially blocked cross-validation is implemented; all metrics reflect in-sample fit.
- Residual spatial clustering suggests omitted variables (grid architecture, industrial composition, policy zones).

**Future work:**
- Incorporate exogenous road-density measures from OpenStreetMap to better isolate infrastructure effects.
- Address nonlinear dynamics in extreme urban cores through spline terms or quantile regression.
- Harmonize tile resolution across countries and test elasticity stability under alternative spatial resolutions.
- Extend analysis across 2014–2023 panel to evaluate temporal stability of elasticities.
- Implement spatially blocked cross-validation for out-of-sample evaluation.

---
## 10. References

Dinkelman, T. (2011). The effects of rural electrification on employment: New evidence from South Africa. *American Economic Review*, *101*(7), 3078–3108. https://doi.org/10.1257/aer.101.7.3078

Falchetta, G., Pachauri, S., Byers, E., Danylo, O., & Parkinson, S. C. (2020). Satellite observations reveal inequalities in the progress and effectiveness of recent electrification in Sub-Saharan Africa. *One Earth*, *2*(4), 364–379. https://doi.org/10.1016/j.oneear.2020.03.007

Henderson, J. V., Storeygard, A., & Weil, D. N. (2012). Measuring economic growth from outer space. *American Economic Review*, *102*(2), 994–1028. https://doi.org/10.1257/aer.102.2.994

Lessmann, C., & Seidel, A. (2017). Regional inequality, convergence, and its determinants – A view from outer space. *European Economic Review*, *92*, 110–132. https://doi.org/10.1016/j.euroecorev.2016.11.009

Lipscomb, M., Mobarak, A. M., & Barham, T. (2013). Development effects of electrification: Evidence from the topographic placement of hydropower plants in Brazil. *American Economic Journal: Applied Economics*, *5*(2), 200–231. https://doi.org/10.1257/app.5.2.200

---

**Datasets and tools:**
- VIIRS nighttime lights annual composites — Earth Observation Group, Colorado School of Mines ("Annual VNL V1"; vcm configuration)
- WorldPop population surfaces (RF/dasymetric redistribution; https://www.worldpop.org)
- Heteroskedasticity-robust inference: `statsmodels` `het_breuschpagan` and `get_robustcov_results`
- Regression metrics: `scikit-learn` R² and RMSE

**GitHub repository:** namibarsbold/STATS201-Course-project

**AI Acknowledgment:** Regressions were brainstormed and edited with assistance from ChatGPT for correctness. Project scaffolding, file organization, and GitHub Pages setup were assisted by Claude Code (Anthropic).